In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

class DynamicAnalyticsWorkstation:
    def __init__(self):
        self.station_id = "SEW-STATION-04"
        self.machine_id = "LOCKSTITCH-JUKI-8700"
        self.operator_id = "OP-RAMESH-K (ID: 409)"
        
        self.state = 'SCAN_INPUT'
        self.bundles = [] 
        self.current_tag = None
        self.output_carrier = None
        
        self.cycle_start_time = time.time()
        self.last_drop_time = time.time()
        self.op_start_time = None
        self.last_op_duration = 0  # Track actual active work time
        
        self.event_ledger = []
        self.log_messages = []
        self.log("Workstation initialized with dynamic utilization calculation.")

    def log(self, msg):
        ts_str = time.strftime('%H:%M:%S')
        self.log_messages.insert(0, f"[{ts_str}] {msg}")
        if len(self.log_messages) > 6:
            self.log_messages.pop()

sim = DynamicAnalyticsWorkstation()

out_flow = widgets.Output()
out_analytics = widgets.Output()
out_controls = widgets.Output()
out_log = widgets.Output()

def render_ui():
    with out_flow:
        clear_output(wait=True)
        print("==========================================================")
        print(f" WORKSTATION: {sim.station_id} | MACHINE: {sim.machine_id}")
        print("==========================================================")
        
        for h in sim.event_ledger:
            print(f"[{h['timestamp_str']}] ✅ {h['action']} | Wait: {h.get('wait_time_sec', 0)}s | Duration: {h.get('duration_sec', 0)}s")
            print("   ------------------------------------------------------")
            
        if sim.state == 'SCAN_INPUT':
            print("📲 [ACTION] Scan incoming bundle RFID tag.")
        elif sim.state == 'INSPECT':
            print(f"🔍 [ACTION] Quality Check for Bundle Tag: [{sim.current_tag}]")
        elif sim.state == 'OPERATION':
            print(f"⚙️ [ACTION] Operation in progress on {len(sim.bundles)} bundles...")
        elif sim.state == 'SCAN_OUTPUT':
            print("📲 [ACTION] Scan output RFID to finish operation.")
        elif sim.state == 'DONE':
            print("📦 [ACTION] Drop bundle into output bin.")
        print("==========================================================")

    with out_analytics:
        clear_output(wait=True)
        print("📊 DYNAMIC ANALYTICS & KPI PANEL (Computed from Timestamps)")
        print("----------------------------------------------------------")
        
        total_cycles = len([e for e in sim.event_ledger if 'Bin Drop' in e['action']])
        
        # Aggregate all past defects stored safely
        all_bundles_list = sim.bundles + getattr(sim, 'past_bundles', [])
        total_defects = sum([b['defects'] for b in all_bundles_list])
        
        avg_wait = int(sum([e.get('wait_time_sec', 0) for e in sim.event_ledger]) / max(1, len(sim.event_ledger)))
        fpy = max(0.0, 100.0 - (total_defects * 10.0))
        
        # DYNAMIC UTILIZATION COMPUTATION: Active sewing time ÷ Total cycle time of latest drop
        last_drop_event = next((e for e in reversed(sim.event_ledger) if 'Bin Drop' in e['action']), None)
        if last_drop_event and last_drop_event.get('total_cycle_time', 0) > 0:
            active_work = last_drop_event.get('op_duration', 0)
            total_t = last_drop_event.get('total_cycle_time', 0)
            machine_util = min(100.0, round((active_work / total_t) * 100.0, 1))
        else:
            machine_util = 0.0 if total_cycles == 0 else 50.0

        print(f" • Total Completed Cycles : {total_cycles}")
        print(f" • First Pass Yield (FPY) : {fpy:.1f}%")
        print(f" • Machine Utilization    : {machine_util}% (Active Work ÷ Total Cycle Time)")
        print(f" • Avg Queue Wait Time    : {avg_wait}s between operations")
        print(f" • Cumulative Defects     : {total_defects} reported")
        print("==========================================================")

    with out_controls:
        clear_output(wait=True)
        if sim.state == 'SCAN_INPUT':
            tag_id = f"TAG-{int(time.time())}"
            btn = widgets.Button(description=f"📲 Scan Bundle ({tag_id})", button_style='info', layout=widgets.Layout(width='350px', height='40px'))
            btn.on_click(lambda b, t=tag_id: scan_input(t))
            display(btn)
            if len(sim.bundles) > 0:
                btn_op = widgets.Button(description="⚙️ Start Operation ➔", button_style='success', layout=widgets.Layout(width='350px', height='40px'))
                btn_op.on_click(lambda b: start_op())
                display(btn_op)
        elif sim.state == 'INSPECT':
            b0 = widgets.Button(description="🟢 Quality OK (0 Defects)", button_style='success', layout=widgets.Layout(width='170px', height='40px'))
            b1 = widgets.Button(description="⚠️ Report 1 Defect", button_style='warning', layout=widgets.Layout(width='170px', height='40px'))
            b0.on_click(lambda b: record_insp(0))
            b1.on_click(lambda b: record_insp(1))
            display(widgets.HBox([b0, b1]))
        elif sim.state == 'OPERATION':
            btn_fin = widgets.Button(description="⚙️ Finish Operation", button_style='primary', layout=widgets.Layout(width='350px', height='40px'))
            btn_fin.on_click(lambda b: finish_op())
            display(btn_fin)
        elif sim.state == 'SCAN_OUTPUT':
            out_tag = f"CARRIER-OUT-{int(time.time())}"
            btn_out = widgets.Button(description=f"📲 Scan Output RFID ({out_tag})", button_style='info', layout=widgets.Layout(width='350px', height='40px'))
            btn_out.on_click(lambda b, t=out_tag: scan_out(t))
            display(btn_out)
        elif sim.state == 'DONE':
            btn_bin = widgets.Button(description="📥 Drop into Output Bin & Reset", button_style='danger', layout=widgets.Layout(width='350px', height='40px'))
            btn_bin.on_click(lambda b: drop_bin())
            display(btn_bin)

    with out_log:
        clear_output(wait=True)
        print("--- Immutable Event Outbox Ledger ---")
        for m in sim.log_messages:
            print(f" • {m}")

def scan_input(tag_id):
    sim.current_tag = tag_id
    sim.state = 'INSPECT'
    sim.inspect_start_time = time.time()
    sim.log(f"Bundle RFID [{tag_id}] scanned at input.")
    render_ui()

def record_insp(defects):
    duration = int(time.time() - getattr(sim, 'inspect_start_time', time.time()))
    wait_time = int(time.time() - sim.last_drop_time)
    
    sim.bundles.append({"tag": sim.current_tag, "defects": defects})
    sim.event_ledger.append({
        "timestamp_str": time.strftime('%H:%M:%S'),
        "action": f"Scanned [{sim.current_tag}] & QC Checked (Defects: {defects})",
        "wait_time_sec": wait_time,
        "duration_sec": duration
    })
    sim.log(f"Bundle [{sim.current_tag}] inspected in {duration}s. Defects: {defects}.")
    sim.current_tag = None
    sim.state = 'SCAN_INPUT'
    render_ui()

def start_op():
    sim.state = 'OPERATION'
    sim.op_start_time = time.time()
    sim.log(f"Operation started for {len(sim.bundles)} bundles.")
    render_ui()

def finish_op():
    sim.last_op_duration = int(time.time() - sim.op_start_time)
    sim.state = 'SCAN_OUTPUT'
    sim.log(f"Operation completed in {sim.last_op_duration}s. Awaiting output RFID scan.")
    render_ui()

def scan_out(out_tag):
    sim.output_carrier = out_tag
    sim.state = 'DONE'
    sim.log(f"Output RFID carrier [{out_tag}] scanned.")
    render_ui()

def drop_bin():
    drop_time = time.time()
    total_cycle_time = int(drop_time - sim.cycle_start_time)
    
    sim.event_ledger.append({
        "timestamp_str": time.strftime('%H:%M:%S'),
        "action": f"Bin Drop [{sim.output_carrier}]",
        "wait_time_sec": 0,
        "duration_sec": total_cycle_time,
        "op_duration": sim.last_op_duration,
        "total_cycle_time": total_cycle_time
    })
    sim.log(f"Carrier [{sim.output_carrier}] dropped into output bin. Total cycle time: {total_cycle_time}s.")
    
    sim.past_bundles = getattr(sim, 'past_bundles', []) + sim.bundles
    
    sim.bundles = []
    sim.current_tag = None
    sim.output_carrier = None
    sim.cycle_start_time = time.time()
    sim.last_drop_time = drop_time
    sim.last_op_duration = 0
    sim.state = 'SCAN_INPUT'
    render_ui()

display(out_flow, out_analytics, out_controls, out_log)
render_ui()

Output()

Output()

Output()

Output()